In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
os.chdir(r"C:\Users\Mohan2\OneDrive - University of Ottawa\THESIS\test_set")

In [3]:
fragment_df = pd.read_pickle("fragment_database.pkl")

In [13]:
fragment_df

,domain,sufam,group,start,end,PDB
0,d2fkna_,e.51.1,0,137,195,2FKN_A
1,d1leha1,c.2.1,0,18,77,1LEH_A
2,d2fkna_,e.51.1,0,137,200,2FKN_A
3,d1v8ba1,c.2.1,0,3,66,1V8B_A
4,d3vpxa2,c.2.1,0,9,71,3VPX_A
...,...,...,...,...,...,...
24994,d4tv5a_,c.1.12,0,1,91,4TV5_A
24995,d3cz5a_,c.23.1,0,15,104,3CZ5_A
24996,d4tzka_,c.2.1,0,7,133,4TZK_A
24997,d4tzka_,c.2.1,0,3,102,4TZK_A


In [14]:
MLP_predictions = np.load("MLP_predictions.npz")
HMM_predictions = np.load("HMM_predictions.npz") 
PLM_predictions = np.load("PLM_predictions.npz")
PSSM_predictions= np.load("PSSM_predictions.npz")
all_chains = np.load("all_test_chains.npz")

In [15]:
fragment_df = fragment_df[fragment_df['PDB'].isin(all_chains.keys())]
fragment_df

,domain,sufam,group,start,end,PDB
4,d3vpxa2,c.2.1,0,9,71,3VPX_A
7,d3pwza2,c.2.1,1,1,76,3PWZ_A
8,d4omua2,c.2.1,0,1,58,4OMU_A
10,d3dona2,c.2.1,2,1,78,3DON_A
11,d1v76a_,b.137.1,3,15,37,1V76_A
...,...,...,...,...,...,...
24991,d2goja2,d.44.1,760,36,57,2GOJ_A
24993,d3pwza1,c.58.1,0,4,75,3PWZ_A
24994,d4tv5a_,c.1.12,0,1,91,4TV5_A
24996,d4tzka_,c.2.1,0,7,133,4TZK_A


In [7]:
AA_dict = {i:j for i,j in zip(['a','r','n','d','c','q','e','g','h','i','l','k','m','f','p','s','t','w','y','v','x'],range(21))}
AA_dict['X'] = -9

astral_file = open("astral-scopedom-seqres-gd-all-2.08-stable.fa", "r")
domain_dict = {}

line = next(astral_file)
dom_name = line.split()[0][1:]
seq = ""
domain_list = list(fragment_df['domain'])

for line in astral_file:
    if line[0] == ">":
        if dom_name in domain_list:
            try:
                domain_dict[dom_name] = np.array([AA_dict[x] for x in seq], dtype=np.int8)
                seq = ""
                dom_name = line.split()[0][1:]
            except Exception as e:
                print(dom_name, e, seq)
                seq = ""
                dom_name = line.split()[0][1:]
        else:
            seq = ""
            dom_name = line.split()[0][1:]
    else:
        seq = seq + line.strip()

astral_file.close()

In [16]:
discontinuous = []
for domain in fragment_df['domain'].unique():
    try:
        if -9 in domain_dict[domain]:
            discontinuous.append(domain)
    except:
        discontinuous.append(domain)

fragment_df = fragment_df[~fragment_df['domain'].isin(discontinuous)]
fragment_df

,domain,sufam,group,start,end,PDB
4,d3vpxa2,c.2.1,0,9,71,3VPX_A
7,d3pwza2,c.2.1,1,1,76,3PWZ_A
8,d4omua2,c.2.1,0,1,58,4OMU_A
10,d3dona2,c.2.1,2,1,78,3DON_A
11,d1v76a_,b.137.1,3,15,37,1V76_A
...,...,...,...,...,...,...
24991,d2goja2,d.44.1,760,36,57,2GOJ_A
24993,d3pwza1,c.58.1,0,4,75,3PWZ_A
24994,d4tv5a_,c.1.12,0,1,91,4TV5_A
24996,d4tzka_,c.2.1,0,7,133,4TZK_A


In [17]:
import numba as nb

@nb.jit
def _is_equal_nb(seq, subseq, m, i):
    for j in range(m):
        if seq[i + j] != subseq[j]:
            return False
    return True


@nb.jit
def find_loop_nb(seq, subseq):
    n = len(seq)
    m = len(subseq)
    for i in range(n - m + 1):
        if _is_equal_nb(seq, subseq, m, i):
            yield i

In [18]:
def get_subchain_accuracy(stride, domain_len, prediction, ground_truth):
    prediction = prediction[stride:stride+domain_len]
    ground_truth = ground_truth[stride:stride+domain_len]
    mask = ground_truth!=-9
    prediction = prediction[mask]
    ground_truth = ground_truth[mask]
    assert len(prediction)==len(ground_truth)
    return (sum(prediction==ground_truth)/len(prediction))

In [28]:
HMM_predictions['2LZU_A'][5:32]

array([0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2,
       2, 2, 1, 1, 1], dtype=int64)

In [29]:
all_chains['2LZU_A'][1][5:32]

array([1, 1, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 1, 1,
       1, 1, 2, 2, 2])

In [27]:
domain_dict['d2lzua1'][4:32]

array([11,  6, 16,  4,  0,  0,  4,  5, 11, 16, 19, 18, 14, 12,  6,  1, 10,
       19,  0,  3, 11, 10,  9, 13,  8,  2, 15,  4], dtype=int8)

In [20]:
fragment_df = fragment_df.reindex(columns = ['domain', 'sufam', 'group', 'start', 'end', 'PDB', "MLP_f", "PLM_f", "PSSM_f", "HMM_f", "MLP_d", "PLM_d", "PSSM_d", "HMM_d"])

for row in fragment_df.index:
    PDB_ID = fragment_df.loc[row, 'PDB']
    dom_ID = fragment_df.loc[row, 'domain']
    
    if dom_ID in domain_dict.keys():
        if PDB_ID in all_chains.files:
            try:
                ground_truth = all_chains[PDB_ID][1]                
                chain_seq = all_chains[PDB_ID][0]
                domain_seq= domain_dict[dom_ID]
                domain_len = len(domain_seq)
                
                if domain_len == len(chain_seq):
                    stride = 0
                else:
                    stride = next(find_loop_nb(chain_seq, domain_seq))
            
                fragment_df.at[row, 'MLP_d'] = get_subchain_accuracy(stride, domain_len, MLP_predictions[PDB_ID], ground_truth)
                fragment_df.at[row, 'HMM_d'] = get_subchain_accuracy(stride, domain_len, HMM_predictions[PDB_ID], ground_truth)
                fragment_df.at[row, 'PLM_d'] = get_subchain_accuracy(stride, domain_len, PLM_predictions[PDB_ID], ground_truth)
                
                try:
                    fragment_df.at[row, 'PSSM_d']= get_subchain_accuracy(stride, domain_len, PSSM_predictions[PDB_ID], ground_truth)
                except:
                    pass
                
                start = fragment_df.loc[row, 'start'] - 1
                end   = fragment_df.loc[row, 'end']            
                stride = stride + start
                frag_len = end - start
                
                fragment_df.at[row, 'MLP_f'] = get_subchain_accuracy(stride, frag_len, MLP_predictions[PDB_ID], ground_truth)
                fragment_df.at[row, 'HMM_f'] = get_subchain_accuracy(stride, frag_len, HMM_predictions[PDB_ID], ground_truth)
                fragment_df.at[row, 'PLM_f'] = get_subchain_accuracy(stride, frag_len, PLM_predictions[PDB_ID], ground_truth)
                try:
                    fragment_df.at[row, 'PSSM_f']= get_subchain_accuracy(stride, frag_len, PSSM_predictions[PDB_ID], ground_truth)
                except:
                    pass
                
            except Exception as e:
                print(PDB_ID, e)

3FPZ_A 
1AUT_L 
1PFX_L 
3RQI_A 
3RQI_A 
2IV2_X 
1KQF_A 
1PFX_L 
1Y7L_A 
2Q3B_A 
4H27_A 
1Y7L_A 
2Q3B_A 
4H27_A 
1BBO_A 
1U9C_A 
2Q3B_A 
2CIW_A 
4OB0_A 
4V2P_A 
1JVN_A 
4H27_A 
3JU7_A 
2IV2_X 
2Z10_A 
1KB0_A 
1KB0_A 
1KB0_A 
1KB0_A 
1KB0_A 
1OKG_A 
1J9B_A 
2PSP_A 
2PSP_A 
3QYH_A 
3QYH_A 
1OKG_A 


KeyboardInterrupt: 

In [62]:
fragment_df = fragment_df[fragment_df['MLP_d'].notnull()]
fragment_df.to_csv("fragment.csv")
fragment_df

,domain,sufam,group,start,end,PDB,MLP_f,PLM_f,PSSM_f,HMM_f,MLP_d,PLM_d,PSSM_d,HMM_d
4,d3vpxa2,c.2.1,0,9,71,3VPX_A,0.730159,0.84127,0.84127,0.761905,0.547945,0.840183,0.821918,0.488584
7,d3pwza2,c.2.1,1,1,76,3PWZ_A,0.486842,0.868421,0.828947,0.565789,0.614035,0.906433,0.818713,0.584795
8,d4omua2,c.2.1,0,1,58,4OMU_A,0.551724,0.87931,0.810345,0.431034,0.655172,0.867816,0.793103,0.557471
10,d3dona2,c.2.1,2,1,78,3DON_A,0.487179,0.846154,0.679487,0.346154,0.570552,0.895706,0.766871,0.417178
11,d1v76a_,b.137.1,3,15,37,1V76_A,0.565217,0.826087,0.695652,0.434783,0.586957,0.836957,0.728261,0.423913
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24991,d2goja2,d.44.1,760,36,57,2GOJ_A,0.727273,0.909091,0.818182,0.454545,0.72807,0.850877,0.798246,0.614035
24993,d3pwza1,c.58.1,0,4,75,3PWZ_A,0.666667,0.902778,0.819444,0.583333,0.61,0.9,0.81,0.55
24994,d4tv5a_,c.1.12,0,1,91,4TV5_A,0.747253,0.956044,0.901099,0.549451,0.706122,0.934694,0.881633,0.542857
24996,d4tzka_,c.2.1,0,7,133,4TZK_A,0.661417,0.92126,0.850394,0.503937,0.712687,0.910448,0.802239,0.548507


In [65]:
def get_notfrag(stride, domain_len, prediction, ground_truth):
    mask = np.zeros(prediction.shape, dtype=bool)
    mask[stride:stride+domain_len] = True
    prediction = prediction[~mask]
    ground_truth = ground_truth[~mask]
    
    mask = ground_truth!=-9
    prediction = prediction[mask]
    ground_truth = ground_truth[mask]
    
    assert len(prediction)==len(ground_truth)
    return (sum(prediction==ground_truth)/len(prediction))

In [67]:
fragment_df['MLP_d'] = None
fragment_df['PLM_d'] = None
fragment_df['PSSM_d']= None
fragment_df['HMM_d'] = None

for row in fragment_df.index:
    PDB_ID = fragment_df.loc[row, 'PDB']
    dom_ID = fragment_df.loc[row, 'domain']
    
    if dom_ID in domain_dict.keys():
        if PDB_ID in all_chains.files:
            try:
                ground_truth = all_chains[PDB_ID][1]                
                chain_seq = all_chains[PDB_ID][0]
                domain_seq= domain_dict[dom_ID]
                domain_len = len(domain_seq)
                
                if domain_len == len(chain_seq):
                    stride = 0
                else:
                    stride = next(find_loop_nb(chain_seq, domain_seq))
                        
                start = fragment_df.loc[row, 'start'] - 1
                end   = fragment_df.loc[row, 'end']            
                stride = stride + start
                frag_len = end - start
                
                fragment_df.at[row, 'MLP_d'] = get_notfrag(stride, frag_len, MLP_predictions[PDB_ID], ground_truth)
                fragment_df.at[row, 'HMM_d'] = get_notfrag(stride, frag_len, HMM_predictions[PDB_ID], ground_truth)
                fragment_df.at[row, 'PLM_d'] = get_notfrag(stride, frag_len, PLM_predictions[PDB_ID], ground_truth)
                
                try:
                    fragment_df.at[row, 'PSSM_d']= get_notfrag(stride, frag_len, PSSM_predictions[PDB_ID], ground_truth)
                except Exception as e:
                    print(PDB_ID, e)

                
            except Exception as e:
                print(PDB_ID, e)

C:\Users\Mohan\AppData\Local\Temp\ipykernel_12896\2670815784.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fragment_df['MLP_d'] = None
C:\Users\Mohan\AppData\Local\Temp\ipykernel_12896\2670815784.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fragment_df['PLM_d'] = None
C:\Users\Mohan\AppData\Local\Temp\ipykernel_12896\2670815784.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveat

2J3S_A '2J3S_A is not a file in the archive'
1UAR_A '1UAR_A is not a file in the archive'
2ANR_A '2ANR_A is not a file in the archive'
2CTE_A '2CTE_A is not a file in the archive'
1J4W_A '1J4W_A is not a file in the archive'
2ANR_A '2ANR_A is not a file in the archive'
2CTK_A '2CTK_A is not a file in the archive'
1J4W_A '1J4W_A is not a file in the archive'
1TUA_A '1TUA_A is not a file in the archive'
2E3U_A '2E3U_A is not a file in the archive'
2JE6_I '2JE6_I is not a file in the archive'
1TUA_A '1TUA_A is not a file in the archive'
2Z0S_A '2Z0S_A is not a file in the archive'
2E3U_A '2E3U_A is not a file in the archive'
1ATA_A '1ATA_A is not a file in the archive'
1IHU_A '1IHU_A is not a file in the archive'
1M3V_A '1M3V_A is not a file in the archive'
1B8T_A '1B8T_A is not a file in the archive'
1KZY_C '1KZY_C is not a file in the archive'
1CWV_A '1CWV_A is not a file in the archive'
1F00_I '1F00_I is not a file in the archive'
2EW9_A '2EW9_A is not a file in the archive'
2EW9_A '2E

3GQI_B '3GQI_B is not a file in the archive'
1TUA_A '1TUA_A is not a file in the archive'
1HH2_P '1HH2_P is not a file in the archive'
4UDG_A '4UDG_A is not a file in the archive'
4UDG_A '4UDG_A is not a file in the archive'
1OKG_A '1OKG_A is not a file in the archive'
1URH_A '1URH_A is not a file in the archive'
1UAR_A '1UAR_A is not a file in the archive'
2CTF_A '2CTF_A is not a file in the archive'
2ANR_A '2ANR_A is not a file in the archive'
2CTK_A '2CTK_A is not a file in the archive'
1TUA_A '1TUA_A is not a file in the archive'
2CTE_A '2CTE_A is not a file in the archive'
2E3U_A '2E3U_A is not a file in the archive'
1CWV_A '1CWV_A is not a file in the archive'
3RKP_A '3RKP_A is not a file in the archive'
1CWV_A '1CWV_A is not a file in the archive'
3RKP_A '3RKP_A is not a file in the archive'
2CTF_A '2CTF_A is not a file in the archive'
2CTE_A '2CTE_A is not a file in the archive'
1J4W_A '1J4W_A is not a file in the archive'
1TUA_A '1TUA_A is not a file in the archive'
1URH_A '1U

1H0Z_A '1H0Z_A is not a file in the archive'
1R0R_I '1R0R_I is not a file in the archive'
1TBR_R '1TBR_R is not a file in the archive'
1TBR_R '1TBR_R is not a file in the archive'
3ELG_A '3ELG_A is not a file in the archive'
1QKS_A '1QKS_A is not a file in the archive'
1FNF_A '1FNF_A is not a file in the archive'
3ERB_A '3ERB_A is not a file in the archive'
1LY2_A '1LY2_A is not a file in the archive'
1GKN_A '1GKN_A is not a file in the archive'
1QUB_A '1QUB_A is not a file in the archive'
2QY0_A '2QY0_A is not a file in the archive'
1H03_P '1H03_P is not a file in the archive'
2DIB_A '2DIB_A is not a file in the archive'
2DIB_A '2DIB_A is not a file in the archive'
1H6L_A '1H6L_A is not a file in the archive'
1TL2_A '1TL2_A is not a file in the archive'
2CTK_A '2CTK_A is not a file in the archive'
1J4W_A '1J4W_A is not a file in the archive'
1UAR_A '1UAR_A is not a file in the archive'
1VK3_A '1VK3_A is not a file in the archive'
2EBF_X '2EBF_X is not a file in the archive'
1NR0_A '1N

2CTF_A '2CTF_A is not a file in the archive'
2CTF_A '2CTF_A is not a file in the archive'
2CTF_A '2CTF_A is not a file in the archive'
2EBF_X '2EBF_X is not a file in the archive'
5FKT_A '5FKT_A is not a file in the archive'
1JOF_A '1JOF_A is not a file in the archive'
1IJQ_A '1IJQ_A is not a file in the archive'
1JOF_A '1JOF_A is not a file in the archive'
1Q7F_A '1Q7F_A is not a file in the archive'
1JOF_A '1JOF_A is not a file in the archive'
1JOF_A '1JOF_A is not a file in the archive'
1JOF_A '1JOF_A is not a file in the archive'
1JOF_A '1JOF_A is not a file in the archive'
3AMR_A '3AMR_A is not a file in the archive'
1JOF_A '1JOF_A is not a file in the archive'
3AMR_A '3AMR_A is not a file in the archive'
4G1M_A '4G1M_A is not a file in the archive'
4WJK_A '4WJK_A is not a file in the archive'
4Y0G_A division by zero
1W8O_A '1W8O_A is not a file in the archive'
4UDG_A '4UDG_A is not a file in the archive'
1W8O_A '1W8O_A is not a file in the archive'
1W8O_A '1W8O_A is not a file in

In [87]:
# fragment_df[fragment_df['MLP_d'].isnull()]
# index = fragment_df[fragment_df['MLP_d'].isnull()].index
index = [3438, 21230, 22158]
fragment_df.loc[index, "PLM_d"] = fragment_df.loc[index, "PLM_f" ]
fragment_df.loc[index, "PSSM_d"] = fragment_df.loc[index, "PSSM_f" ]
fragment_df.loc[index, "HMM_d"] = fragment_df.loc[index, "HMM_f" ]
fragment_df.loc[index] 

,domain,sufam,group,start,end,PDB,MLP_f,PLM_f,PSSM_f,HMM_f,MLP_d,PLM_d,PSSM_d,HMM_d
3438,d1spwa_,g.41.5,42,1,39,1SPW_A,0.794872,0.74359,0.897436,0.384615,0.794872,0.74359,0.897436,0.384615
21230,d2qv0a_,c.23.1,0,1,122,2QV0_A,0.721311,0.909836,0.860656,0.565574,0.721311,0.909836,0.860656,0.565574
22158,d4y0ga_,b.1.18,792,1,82,4Y0G_A,0.646341,0.841463,0.780488,0.560976,0.646341,0.841463,0.780488,0.560976


In [88]:
fragment_df[fragment_df['PSSM_f'].isnull()]

,domain,sufam,group,start,end,PDB,MLP_f,PLM_f,PSSM_f,HMM_f,MLP_d,PLM_d,PSSM_d,HMM_d
131,d2j3sa2,b.1.18,24,57,84,2J3S_A,0.681818,0.727273,None,0.5,0.685484,0.814516,None,0.580645
148,d1uara2,c.46.1,0,86,116,1UAR_A,0.451613,0.774194,None,0.645161,0.632653,0.853061,None,0.538776
838,d2anra1,d.51.1,36,2,36,2ANR_A,0.485714,0.857143,None,0.285714,0.708333,0.908333,None,0.666667
839,d2ctea1,d.51.1,36,9,44,2CTE_A,0.527778,0.75,None,0.5,0.689655,0.896552,None,0.706897
846,d1j4wa1,d.51.1,36,1,33,1J4W_A,0.363636,0.69697,None,0.333333,0.741071,0.848214,None,0.5625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24874,d2d7ma1,b.1.18,71,19,80,2D7M_A,0.741935,0.903226,None,0.596774,0.830189,0.924528,None,0.698113
24984,d4xyha_,b.69.4,172,119,338,4XYH_A,0.55,0.859091,None,0.368182,0.619355,0.864516,None,0.406452
24986,d4xyha_,b.69.4,1080,135,340,4XYH_A,0.558252,0.864078,None,0.349515,0.60355,0.857988,None,0.426036
24987,d4xyha_,b.69.4,482,162,340,4XYH_A,0.541899,0.905028,None,0.340782,0.612245,0.821429,None,0.423469


In [89]:
fragment_df.to_csv("fragment1.csv")

In [64]:
x= np.arange(20)
mask = np.zeros(x.shape,dtype=bool)
mask[1:5] = True
x[~mask]

array([ 0,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19])

In [50]:
domain_dict['d3hzha_']

array([15, 11, 14,  1,  7,  9,  2, 18,  3, 16,  7,  9, 14, 13,  2, 19, 10,
        9, 19,  3,  3, 15, 19, 13, 16, 19, 11,  5, 10, 16,  5,  9, 13, 16,
       15,  6,  7, 13,  2,  9,  9,  3, 16,  0,  0,  3,  7,  6,  6,  0, 19,
        9, 11, 18, 11,  2,  8, 18, 14,  2,  9,  3,  9, 19, 16, 10,  3,  9,
       16, 12, 14, 11, 12,  3,  7,  9, 16,  4, 10, 15,  2,  9, 12,  6, 13,
        3, 11,  2,  0,  1, 19,  9, 12,  9, 15,  0, 10,  7, 11,  6,  5, 10,
       19, 11,  3,  4, 10,  9, 11,  7,  0, 11, 16, 13,  9, 19, 11, 14, 10,
        3,  1,  0, 11, 19, 10,  5,  1, 19, 12, 15, 19, 13, 19, 11],
      dtype=int8)

In [55]:
all_chains['3HZH_A'][0][23:]

array([15, 11, 14,  1,  7,  9,  2, 18,  3, 16,  7,  9, 14, 13,  2, 19, 10,
        9, 19,  3,  3, 15, 19, 13, 16, 19, 11,  5, 10, 16,  5,  9, 13, 16,
       15,  6,  7, 13,  2,  9,  9,  3, 16,  0,  0,  3,  7,  6,  6,  0, 19,
        9, 11, 18, 11,  2,  8, 18, 14,  2,  9,  3,  9, 19, 16, 10, 20,  9,
       16, 12, 14, 11, 12,  3,  7,  9, 16,  4, 10, 15,  2,  9, 12,  6, 13,
        3, 11,  2,  0,  1, 19,  9, 12,  9, 15,  0, 10,  7, 11,  6,  5, 10,
       19, 11,  3,  4, 10,  9, 11,  7,  0, 11, 16, 13,  9, 19, 11, 14, 10,
        3,  1,  0, 11, 19, 10,  5,  1, 19, 12, 15, 19, 13, 19, 11])

In [60]:
domain_dict['d3hzha_'][66:]

array([ 3,  9, 16, 12, 14, 11, 12,  3,  7,  9, 16,  4, 10, 15,  2,  9, 12,
        6, 13,  3, 11,  2,  0,  1, 19,  9, 12,  9, 15,  0, 10,  7, 11,  6,
        5, 10, 19, 11,  3,  4, 10,  9, 11,  7,  0, 11, 16, 13,  9, 19, 11,
       14, 10,  3,  1,  0, 11, 19, 10,  5,  1, 19, 12, 15, 19, 13, 19, 11],
      dtype=int8)

In [52]:
MLP_predictions['3HZH_A']

array([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2,
       2, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2, 1, 1, 1, 0, 0,
       0, 2, 2, 2, 0, 0, 0, 0, 0, 1, 0, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1,
       1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       2, 2, 2, 0, 0, 1, 1, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 2, 2], dtype=int64)

In [53]:
next(find_loop_nb(all_chains['3HZH_A'][0], domain_dict['d3hzha_']))

StopIteration: 